# LeetCode 908 — Smallest Range I bằng Python

**Link đề:** https://leetcode.com/problems/smallest-range-i/
**Độ khó:** Easy · **Chủ đề:** Array, Math

```python
class Solution:
    def smallestRangeI(self, nums: list[int], k: int) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ đề "mỗi phần tử được cộng một số `x ∈ [-k, k]`, tối đa một lần" và định nghĩa **score = max − min**.
2. Thấy rằng **chỉ phần tử lớn nhất và nhỏ nhất** quyết định đáp án → công thức `max(0, (max − k) − (min + k))`.
3. Đi từ **vét cạn mọi lựa chọn** (để kiểm chứng) → tư duy **khoảng** `[a − k, a + k]` và **giao các khoảng** → công thức `O(n)`.
4. Hiểu vì sao phải kẹp với `0` (`max(0, ...)`), trace, chứng minh hai chiều (cận dưới + xây dựng được), test đối chiếu.
5. Phân biệt với bài khó hơn **910 Smallest Range II** (bắt buộc ±k), luyện 6 bài tập.

## 1. Đề bài và ví dụ

Cho mảng `nums` và số nguyên `k`. Với **mỗi** chỉ số `i`, được làm **tối đa một lần**: đổi `nums[i]` thành `nums[i] + x`, với `x` là số nguyên trong `[-k, k]`.

**Score** của mảng = `max(nums) − min(nums)`. Trả về **score nhỏ nhất** có thể đạt được.

| nums | k | Output | Giải thích |
|---|---:|---:|---|
| `[1]` | 0 | 0 | 1 phần tử: max = min |
| `[0,10]` | 2 | 6 | đổi thành `[2, 8]` → 8 − 2 |
| `[1,3,6]` | 3 | 0 | đổi thành `[4, 4, 4]` |

**Constraints:** `1 <= nums.length <= 10^4`, `0 <= nums[i] <= 10^4`, `0 <= k <= 10^4`.

"Tối đa một lần" + `x` có thể bằng `0` ⇒ thực chất mỗi phần tử có thể trở thành **bất kỳ số nguyên nào** trong đoạn `[nums[i] − k, nums[i] + k]`.

## 2. Nền tảng Python và tư duy cần biết

### 2.1. `max()`, `min()` của list

`max(nums)`, `min(nums)` duyệt toàn bộ list, `O(n)`. Score của một mảng là `max(nums) - min(nums)`.

In [1]:
nums = [1, 3, 6]
print("max =", max(nums), "| min =", min(nums), "| score =", max(nums) - min(nums))

max = 6 | min = 1 | score = 5


### 2.2. `range(a - k, a + k + 1)` — mọi giá trị một phần tử có thể trở thành

`range(start, stop)` **không bao gồm** `stop`, nên muốn lấy đến `a + k` phải viết `a + k + 1`.

In [2]:
a, k = 5, 2
print("Các giá trị có thể của a=5 với k=2:", list(range(a - k, a + k + 1)))

Các giá trị có thể của a=5 với k=2: [3, 4, 5, 6, 7]


### 2.3. `itertools.product` — mọi tổ hợp lựa chọn

`product(A, B, C)` sinh mọi bộ `(a, b, c)` với `a ∈ A`, `b ∈ B`, `c ∈ C` — tương đương các vòng `for` lồng nhau với số tầng tùy ý. Ta dùng nó để **vét cạn** cho mảng nhỏ.

In [3]:
from itertools import product
print(list(product([1, 2], ["x", "y"])))
print("Số tổ hợp với 3 phần tử, mỗi phần tử 5 lựa chọn:", len(list(product(range(5), repeat=3))))

[(1, 'x'), (1, 'y'), (2, 'x'), (2, 'y')]
Số tổ hợp với 3 phần tử, mỗi phần tử 5 lựa chọn: 125


### 2.4. Tư duy "khoảng" (interval)

Mỗi phần tử `a` trở thành một **khoảng** các giá trị có thể: `[a − k, a + k]`. Câu hỏi "có thể làm mọi phần tử **bằng nhau** không?" ⇔ "các khoảng có **điểm chung** không?".

Giao của nhiều khoảng `[L_i, R_i]` là `[max(L_i), min(R_i)]` — khác rỗng khi `max(L_i) <= min(R_i)`.

In [4]:
intervals = [(1 - 3, 1 + 3), (3 - 3, 3 + 3), (6 - 3, 6 + 3)]
print("Khoảng của [1,3,6] với k=3:", intervals)
L = max(l for l, r in intervals)
R = min(r for l, r in intervals)
print(f"Giao = [{L}, {R}] ->", "có điểm chung" if L <= R else "rỗng")

Khoảng của [1,3,6] với k=3: [(-2, 4), (0, 6), (3, 9)]
Giao = [3, 4] -> có điểm chung


## 3. Cách 1 — Vét cạn mọi lựa chọn (chỉ để hiểu và kiểm chứng)

Thử mọi cách chọn `x_i ∈ [-k, k]` cho từng phần tử, tính score, lấy nhỏ nhất.

Số tổ hợp `(2k + 1)^n` — bùng nổ cực nhanh. Chỉ dùng cho `n`, `k` rất nhỏ làm **oracle** (đáp án chuẩn) để test các cách khác.

In [5]:
def smallest_range_bruteforce(nums: list[int], k: int) -> int:
    choices = [range(a - k, a + k + 1) for a in nums]
    best = None
    for combo in product(*choices):
        score = max(combo) - min(combo)
        if best is None or score < best:
            best = score
    return best

print(smallest_range_bruteforce([1], 0))
print(smallest_range_bruteforce([0, 10], 2))
print(smallest_range_bruteforce([1, 3, 6], 3))
print("Số tổ hợp với [0,10], k=2:", (2 * 2 + 1) ** 2)

0
6
0
Số tổ hợp với [0,10], k=2: 25


### 3.1. Nút thắt

`n = 10^4`, `k = 10^4` → `(20001)^(10^4)` tổ hợp: không thể. Phải **suy luận** thay vì thử.

Quan sát kết quả vét cạn: đáp án chỉ phụ thuộc vào **max và min** của mảng gốc? Hãy thử vài mảng có cùng max/min nhưng phần giữa khác nhau.

In [6]:
for arr in ([0, 10], [0, 5, 10], [0, 1, 9, 10], [0, 10, 10, 10]):
    print(arr, "k=2 ->", smallest_range_bruteforce(arr, 2))

[0, 10] k=2 -> 6
[0, 5, 10] k=2 -> 6
[0, 1, 9, 10] k=2 -> 6
[0, 10, 10, 10] k=2 -> 6


Tất cả đều ra **6**! Phần tử ở giữa không ảnh hưởng. Đây là manh mối dẫn tới insight.

## 4. Insight: chỉ max và min quyết định đáp án

Gọi `mn = min(nums)`, `mx = max(nums)`.

**Muốn score nhỏ** → kéo các giá trị lại gần nhau: **tăng** phần tử nhỏ nhất hết mức (`mn + k`), **giảm** phần tử lớn nhất hết mức (`mx − k`).

- Nếu `mx − k <= mn + k` (tức `mx − mn <= 2k`): mọi khoảng `[a − k, a + k]` đều chứa đoạn `[mx − k, mn + k]` (vì `a − k ≤ mx − k` và `a + k ≥ mn + k`) → chọn **cùng một giá trị** cho tất cả → score **0**.
- Ngược lại: score tốt nhất là `(mx − k) − (mn + k) = mx − mn − 2k`.

Gộp lại:

```text
answer = max(0, mx − mn − 2k)
```

Kiểm tra: `[0,10], k=2` → `10 − 0 − 4 = 6` ✔. `[1,3,6], k=3` → `6 − 1 − 6 = −1` → kẹp về `0` ✔.

In [7]:
for nums, k in [([1], 0), ([0, 10], 2), ([1, 3, 6], 3)]:
    mx, mn = max(nums), min(nums)
    raw = mx - mn - 2 * k
    print(f"nums={nums}, k={k}: mx-mn-2k = {mx}-{mn}-{2*k} = {raw} -> max(0, {raw}) = {max(0, raw)}")

nums=[1], k=0: mx-mn-2k = 1-1-0 = 0 -> max(0, 0) = 0
nums=[0, 10], k=2: mx-mn-2k = 10-0-4 = 6 -> max(0, 6) = 6
nums=[1, 3, 6], k=3: mx-mn-2k = 6-1-6 = -1 -> max(0, -1) = 0


## 5. Cách 2 — Giao các khoảng (cài đặt trực tiếp từ tư duy khoảng)

Tính `L = max(a − k)` và `R = min(a + k)` trên mọi phần tử:
- `L <= R` → có điểm chung → `0`.
- `L > R` → khoảng cách nhỏ nhất giữa "đầu trái lớn nhất" và "đầu phải nhỏ nhất" là `L − R`.

Thật ra `L = mx − k` và `R = mn + k`, nên `L − R = mx − mn − 2k` — trùng công thức.

In [8]:
def smallest_range_intervals(nums: list[int], k: int) -> int:
    L = max(a - k for a in nums)      # đầu trái lớn nhất
    R = min(a + k for a in nums)      # đầu phải nhỏ nhất
    return 0 if L <= R else L - R

for nums, k in [([1], 0), ([0, 10], 2), ([1, 3, 6], 3)]:
    print(nums, k, "->", smallest_range_intervals(nums, k))

[1] 0 -> 0
[0, 10] 2 -> 6
[1, 3, 6] 3 -> 0


## 6. Cách 3 — Sắp xếp rồi lấy hai đầu

Nếu chưa quen `max`/`min`, có thể sắp xếp: phần tử đầu là min, phần tử cuối là max. Đúng nhưng `O(n log n)` — chậm hơn cần thiết.

In [9]:
def smallest_range_sort(nums: list[int], k: int) -> int:
    s = sorted(nums)
    return max(0, s[-1] - s[0] - 2 * k)

print(smallest_range_sort([0, 10], 2), smallest_range_sort([1, 3, 6], 3))

6 0


`s[-1]` là phần tử **cuối** (chỉ số âm đếm từ cuối), `s[0]` là phần tử đầu.

## 7. Cách 4 — Lời giải chính: một lượt tìm min và max

`max(nums)` và `min(nums)` là hai lượt duyệt. Có thể gộp thành **một** lượt tự viết — minh họa cách tìm cực trị thủ công.

In [10]:
def smallest_range_one_pass(nums: list[int], k: int) -> int:
    mn = mx = nums[0]
    for a in nums[1:]:
        if a < mn:
            mn = a
        elif a > mx:
            mx = a
    return max(0, mx - mn - 2 * k)

for nums, k in [([1], 0), ([0, 10], 2), ([1, 3, 6], 3)]:
    print(nums, k, "->", smallest_range_one_pass(nums, k))

[1] 0 -> 0
[0, 10] 2 -> 6
[1, 3, 6] 3 -> 0


Dùng `elif` an toàn: nếu `a < mn` thì `a` không thể đồng thời `> mx` (vì `mn <= mx`).

### 7.1. Xây dựng **cụ thể** mảng sau khi đổi (không chỉ con số)

Chứng minh tốt nhất là **chỉ ra cách đổi**. Chọn mục tiêu:
- Nếu đáp án là 0: chọn `t` bất kỳ trong `[mx − k, mn + k]` (ví dụ `t = mx − k`), đưa mọi phần tử về `t`.
- Nếu không: đưa mỗi phần tử về giá trị gần nhất trong đoạn `[mn + k, mx − k]` — phần tử nhỏ được tăng, lớn được giảm, nhưng không ai vượt quá đoạn.

`min(max(v, lo), hi)` là phép **kẹp** (clamp) `v` vào đoạn `[lo, hi]`.

In [11]:
def clamp(v, lo, hi):
    return min(max(v, lo), hi)

def construct(nums, k):
    mn, mx = min(nums), max(nums)
    if mx - mn <= 2 * k:
        t = mx - k
        new = [t] * len(nums)
    else:
        lo, hi = mn + k, mx - k
        new = [clamp(a, lo, hi) for a in nums]
    assert all(abs(n - a) <= k for n, a in zip(new, nums)), "mỗi thay đổi phải trong [-k, k]"
    return new, max(new) - min(new)

for nums, k in [([0, 10], 2), ([1, 3, 6], 3), ([0, 5, 10], 2), ([4, 9, 1, 7], 1)]:
    new, score = construct(nums, k)
    print(f"nums={nums}, k={k} -> đổi thành {new}, score = {score}")

nums=[0, 10], k=2 -> đổi thành [2, 8], score = 6
nums=[1, 3, 6], k=3 -> đổi thành [3, 3, 3], score = 0
nums=[0, 5, 10], k=2 -> đổi thành [2, 5, 8], score = 6
nums=[4, 9, 1, 7], k=1 -> đổi thành [4, 8, 2, 7], score = 6


## 8. TRACE từng bước

### 8.1. Bảng trace cách 4 với `nums = [4, 9, 1, 7]`, `k = 1`

| a | a < mn? | a > mx? | mn | mx |
|---:|:---:|:---:|---:|---:|
| (khởi tạo 4) | | | 4 | 4 |
| 9 | không | **có** | 4 | 9 |
| 1 | **có** | — | 1 | 9 |
| 7 | không | không | 1 | 9 |

Kết thúc: `max(0, 9 − 1 − 2·1) = 6`.

In [12]:
def trace_smallest_range(nums, k):
    mn = mx = nums[0]
    print(f"Khởi tạo mn = mx = {nums[0]}")
    for a in nums[1:]:
        if a < mn:
            mn = a; note = "cập nhật mn"
        elif a > mx:
            mx = a; note = "cập nhật mx"
        else:
            note = "không đổi"
        print(f"  a={a}: {note:12} | mn={mn}, mx={mx}")
    raw = mx - mn - 2 * k
    print(f"mx - mn - 2k = {mx} - {mn} - {2 * k} = {raw} -> đáp án {max(0, raw)}")
    return max(0, raw)

_ = trace_smallest_range([4, 9, 1, 7], 1); print()
_ = trace_smallest_range([1, 3, 6], 3)

Khởi tạo mn = mx = 4
  a=9: cập nhật mx  | mn=4, mx=9
  a=1: cập nhật mn  | mn=1, mx=9
  a=7: không đổi    | mn=1, mx=9
mx - mn - 2k = 9 - 1 - 2 = 6 -> đáp án 6

Khởi tạo mn = mx = 1
  a=3: cập nhật mx  | mn=1, mx=3
  a=6: cập nhật mx  | mn=1, mx=6
mx - mn - 2k = 6 - 1 - 6 = -1 -> đáp án 0


### 8.2. Trace các khoảng — nhìn thấy khi nào giao rỗng

In [13]:
def trace_intervals(nums, k):
    print(f"nums={nums}, k={k}")
    for a in nums:
        print(f"  {a:>2} -> [{a - k:>3}, {a + k:>3}]")
    L, R = max(a - k for a in nums), min(a + k for a in nums)
    if L <= R:
        print(f"  L=max đầu trái={L}, R=min đầu phải={R}: L <= R -> chung điểm, đáp án 0")
    else:
        print(f"  L={L} > R={R}: khoảng hở {L} - {R} = {L - R} -> đáp án {L - R}")

trace_intervals([0, 10], 2); trace_intervals([1, 3, 6], 3)

nums=[0, 10], k=2
   0 -> [ -2,   2]
  10 -> [  8,  12]
  L=8 > R=2: khoảng hở 8 - 2 = 6 -> đáp án 6
nums=[1, 3, 6], k=3
   1 -> [ -2,   4]
   3 -> [  0,   6]
   6 -> [  3,   9]
  L=max đầu trái=3, R=min đầu phải=4: L <= R -> chung điểm, đáp án 0


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def smallestRangeI(self, nums: list[int], k: int) -> int:
        return max(0, max(nums) - min(nums) - 2 * k)

sol = Solution()
print(sol.smallestRangeI([1], 0), sol.smallestRangeI([0, 10], 2), sol.smallestRangeI([1, 3, 6], 3))

0 6 0


### 9.1. Bản nộp một lượt (không dùng hàm có sẵn)

In [15]:
class SolutionOnePass:
    def smallestRangeI(self, nums: list[int], k: int) -> int:
        mn = mx = nums[0]
        for a in nums:
            if a < mn:
                mn = a
            if a > mx:
                mx = a
        diff = mx - mn - 2 * k
        return diff if diff > 0 else 0

print(SolutionOnePass().smallestRangeI([0, 10], 2))

6


## 10. Vì sao đúng? (chứng minh hai chiều)

Gọi `mn`, `mx` là min/max ban đầu, `D = mx − mn`.

**(1) Cận dưới — không thể tốt hơn:** sau khi đổi, phần tử gốc `mx` trở thành ít nhất `mx − k`, phần tử gốc `mn` trở thành nhiều nhất `mn + k`. Score ≥ (giá trị mới của mx) − (giá trị mới của mn) ≥ `(mx − k) − (mn + k) = D − 2k`. Score cũng luôn ≥ 0. ⇒ score ≥ `max(0, D − 2k)`.

**(2) Đạt được — có cách đổi cho đúng giá trị đó** (mục 7.1):
- `D ≤ 2k`: mọi phần tử đưa về `t = mx − k` được, vì `|t − a| ≤ k` với mọi `mn ≤ a ≤ mx` (do `a − k ≤ mx − k = t` và `t = mx − k ≤ mn + k ≤ a + k`). Score = 0.
- `D > 2k`: kẹp mỗi `a` vào `[mn + k, mx − k]`: phần tử nhỏ tăng tối đa `k`, lớn giảm tối đa `k`. Mảng mới có min `= mn + k`, max `= mx − k` → score `= D − 2k`.

(1) + (2) ⇒ đáp án đúng bằng `max(0, D − 2k)`. ∎

## 11. Độ phức tạp và so sánh

`n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ghi chú |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi `x_i` | `O((2k+1)^n · n)` | `O(n)` | Chỉ để kiểm chứng |
| 2. Giao khoảng | `L = max(a−k)`, `R = min(a+k)` | `O(n)` | `O(1)` | Trực quan hình học |
| 3. Sắp xếp | `s[-1] − s[0]` | `O(n log n)` | `O(n)` | Thừa công |
| 4. min/max | `max(0, mx − mn − 2k)` | `O(n)` | `O(1)` | **Khuyên dùng** |

Phải đọc toàn bộ mảng mới biết max/min → `O(n)` là tối ưu.

## 12. Bộ test: mẫu đề, biên, đối chiếu với vét cạn

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

methods = [smallest_range_intervals, smallest_range_sort, smallest_range_one_pass,
           Solution().smallestRangeI, SolutionOnePass().smallestRangeI]

cases = [
    ("đề VD1", [1], 0, 0),
    ("đề VD2", [0, 10], 2, 6),
    ("đề VD3", [1, 3, 6], 3, 0),
    ("k = 0 giữ nguyên", [4, 9, 1, 7], 0, 8),
    ("vừa khít D = 2k", [2, 8], 3, 0),
    ("D = 2k + 1", [2, 9], 3, 1),
    ("mảng hằng", [5, 5, 5], 0, 0),
    ("k rất lớn", [0, 10000], 10000, 0),
    ("biên lớn", [0, 10000], 0, 10000),
]
for name, nums, k, exp in cases:
    for f in methods:
        assert f(nums, k) == exp, (name, f)
    check(name, Solution().smallestRangeI(nums, k), exp)

PASS  đề VD1: 0
PASS  đề VD2: 6
PASS  đề VD3: 0
PASS  k = 0 giữ nguyên: 8
PASS  vừa khít D = 2k: 0
PASS  D = 2k + 1: 1
PASS  mảng hằng: 0
PASS  k rất lớn: 0
PASS  biên lớn: 10000


In [17]:
import random
random.seed(908)
for _ in range(300):
    n = random.randint(1, 4)
    k = random.randint(0, 2)
    nums = [random.randint(0, 8) for _ in range(n)]
    expected = smallest_range_bruteforce(nums, k)
    for f in methods:
        assert f(nums, k) == expected
    new, score = construct(nums, k)
    assert score == expected
print("300 bộ nhỏ ngẫu nhiên: 5 cách + cách xây dựng cụ thể đều khớp vét cạn.")

300 bộ nhỏ ngẫu nhiên: 5 cách + cách xây dựng cụ thể đều khớp vét cạn.


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên kẹp với 0
`mx − mn − 2k` có thể **âm** (ví dụ 3: `−1`). Score không bao giờ âm.

In [18]:
def wrong_no_clamp(nums, k):
    return max(nums) - min(nums) - 2 * k

print("Sai:", wrong_no_clamp([1, 3, 6], 3), "| Đúng:", Solution().smallestRangeI([1, 3, 6], 3))

Sai: -1 | Đúng: 0


### Lỗi 2: Trừ `k` thay vì `2k`
Chỉ chỉnh **một** đầu (giảm max hoặc tăng min). Nhưng cả hai đầu đều được chỉnh → thu hẹp `2k`.

In [19]:
def wrong_one_k(nums, k):
    return max(0, max(nums) - min(nums) - k)

print("Sai:", wrong_one_k([0, 10], 2), "| Đúng:", Solution().smallestRangeI([0, 10], 2))

Sai: 8 | Đúng: 6


### Lỗi 3: Nhầm với 910 Smallest Range II — nghĩ phải cộng **đúng** `+k` hoặc `−k`
Bài này cho **mọi** `x ∈ [−k, k]` (kể cả 0) → linh hoạt hơn nhiều, công thức đơn giản. Bài 910 mới là "bắt buộc ±k" và cần sắp xếp + duyệt điểm cắt.

### Lỗi 4: Tính score sau khi chỉ dịch mọi phần tử **cùng một** lượng
Cộng cùng `x` cho mọi phần tử thì max − min **không đổi** — không giúp gì.

### Lỗi 5: Khởi tạo `mn = 0`, `mx = 0` khi tự tìm min/max
Nếu mọi phần tử > 0 thì `mn` sai (kẹt ở 0). Khởi tạo bằng `nums[0]` (hoặc `float('inf')` / `-float('inf')`).

In [20]:
def wrong_init(nums, k):
    mn = mx = 0                       # SAI
    for a in nums:
        mn = min(mn, a); mx = max(mx, a)
    return max(0, mx - mn - 2 * k)

print("Sai:", wrong_init([7, 9], 0), "| Đúng:", Solution().smallestRangeI([7, 9], 0))
print("Dịch đều +3 không đổi score:", max([x + 3 for x in [0, 10]]) - min([x + 3 for x in [0, 10]]))

Sai: 9 | Đúng: 2
Dịch đều +3 không đổi score: 10


## 14. Mẹo nhận dạng pattern

- Đề hỏi về **max − min** sau khi mỗi phần tử được "xê dịch trong một khoảng" → nghĩ tới **chỉ hai cực trị** quyết định, và tư duy **giao khoảng**.
- Thấy "tối đa một lần", "x trong `[-k, k]`" → mỗi phần tử là một **khoảng** giá trị.
- Kết quả là hiệu/khoảng cách → nhớ **kẹp về 0** (`max(0, ...)`).
- Khi nghi ngờ công thức: viết **vét cạn nhỏ** bằng `itertools.product` làm oracle.

Bài cùng họ: 910 Smallest Range II, 1509 Minimum Difference Between Largest and Smallest Value in Three Moves, 452 Minimum Number of Arrows (giao khoảng).

## 15. Biến thể

### 15.1. LeetCode 910 — Smallest Range II (bắt buộc `+k` hoặc `−k`)
Ý tưởng: sắp xếp; ở một "điểm cắt" `i`, các phần tử `0..i` cộng `k`, phần còn lại trừ `k`. Thử mọi điểm cắt.

In [21]:
def smallest_range_ii(nums: list[int], k: int) -> int:
    s = sorted(nums)
    best = s[-1] - s[0]                          # mọi phần tử cùng dịch
    for i in range(len(s) - 1):
        hi = max(s[i] + k, s[-1] - k)
        lo = min(s[0] + k, s[i + 1] - k)
        best = min(best, hi - lo)
    return best

def smallest_range_ii_brute(nums, k):
    return min(max(c) - min(c) for c in product(*[(a - k, a + k) for a in nums]))

print(smallest_range_ii([1, 3, 6], 3), "| brute:", smallest_range_ii_brute([1, 3, 6], 3))
print(smallest_range_ii([0, 10], 2), "| brute:", smallest_range_ii_brute([0, 10], 2))
random.seed(910)
for _ in range(200):
    nums = [random.randint(0, 9) for _ in range(random.randint(1, 5))]
    k = random.randint(0, 4)
    assert smallest_range_ii(nums, k) == smallest_range_ii_brute(nums, k)
print("910 khớp vét cạn trên 200 bộ ngẫu nhiên.")

3 | brute: 3
6 | brute: 6
910 khớp vét cạn trên 200 bộ ngẫu nhiên.


So sánh: với `[1,3,6], k=3`, bài 908 ra `0` nhưng bài 910 ra `3` — vì 910 không cho phép "cộng một chút".

### 15.2. Mỗi phần tử có `k_i` riêng
Khoảng của phần tử `i` là `[a_i − k_i, a_i + k_i]`; đáp án = `max(0, max(a_i − k_i) − min(a_i + k_i))`. Cách "giao khoảng" (cách 2) mở rộng tự nhiên, còn công thức `mx − mn − 2k` thì **không**.

In [22]:
def smallest_range_individual_k(nums, ks):
    L = max(a - k for a, k in zip(nums, ks))
    R = min(a + k for a, k in zip(nums, ks))
    return max(0, L - R)

print(smallest_range_individual_k([0, 10, 5], [1, 2, 0]))   # khoảng [-1,1],[8,12],[5,5] -> L=8, R=1 -> 7

7


## 16. Bài tập tự luyện

Hàm TODO chỉ định nghĩa, không được gọi khi chạy notebook.

### Bài 1 — Tính tay
`nums = [3, 15, 8, 1]`, `k = 4`. Tính đáp án và chỉ ra một mảng sau khi đổi đạt đáp án đó.

### Bài 2 — Dự đoán output
```python
list(range(5 - 2, 5 + 2))
max(0, 3 - 10)
min(max(12, 4), 9)
```

### Bài 3 — Điền chỗ trống: một lượt min/max

In [23]:
def smallest_range_exercise(nums, k) -> int:
    mn = mx = nums[0]
    for a in nums:
        # TODO: cập nhật mn, mx
        pass
    # TODO: trả về đáp án (nhớ kẹp 0)
    pass

# assert smallest_range_exercise([0, 10], 2) == 6
# assert smallest_range_exercise([1, 3, 6], 3) == 0

### Bài 4 — Tự viết cách "giao khoảng"

In [24]:
def smallest_range_intervals_exercise(nums, k) -> int:
    # TODO: L = max(a - k), R = min(a + k)
    pass

# assert smallest_range_intervals_exercise([4, 9, 1, 7], 1) == 6

### Bài 5 — Biến thể: `k` nhỏ nhất để đạt score 0
Trả về `k` nhỏ nhất (số nguyên ≥ 0) sao cho `smallestRangeI(nums, k) == 0`.

In [25]:
def min_k_for_zero(nums) -> int:
    # TODO: cần mx - mn <= 2k
    pass

# assert min_k_for_zero([1, 3, 6]) == 3
# assert min_k_for_zero([0, 10]) == 5
# assert min_k_for_zero([7]) == 0

### Bài 6 — Mini challenge: trả về **mảng sau khi đổi** đạt score nhỏ nhất
Mỗi phần tử mới phải cách phần tử cũ không quá `k`.

In [26]:
def build_best_array(nums, k) -> list[int]:
    # TODO: dùng ý tưởng kẹp vào [mn + k, mx - k] hoặc đưa hết về mx - k
    pass

# new = build_best_array([4, 9, 1, 7], 1)
# assert all(abs(x - a) <= 1 for x, a in zip(new, [4, 9, 1, 7])) and max(new) - min(new) == 6

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chỉ cần 1 và 15 | `15 − 1 − 8` | 6; kẹp vào `[5, 11]` |
| 2 | range không lấy stop | kẹp 0 | `[3,4,5,6]`, `0`, `9` |
| 3 | so với mn, mx | hai `if` | `return max(0, mx - mn - 2 * k)` |
| 4 | generator trong max/min | so sánh L, R | `0 if L <= R else L - R` |
| 5 | `mx − mn ≤ 2k` | làm tròn lên | `(mx - mn + 1) // 2` |
| 6 | hai trường hợp | `clamp` | `[min(max(a, lo), hi) for a in nums]` |

## 18. Đáp án tham khảo

**Bài 1:** `mn = 1`, `mx = 15` → `15 − 1 − 8 = 6`. Đoạn kẹp `[1 + 4, 15 − 4] = [5, 11]`: `[3,15,8,1] → [5, 11, 8, 5]`, score `11 − 5 = 6`.

**Bài 2:** `[3, 4, 5, 6]`, `0`, `9`.

**Bài 5:** cần `2k ≥ mx − mn` ⇒ `k = ⌈(mx − mn) / 2⌉ = (mx − mn + 1) // 2`.

In [27]:
def ans_smallest_range(nums, k):
    mn = mx = nums[0]
    for a in nums:
        if a < mn: mn = a
        if a > mx: mx = a
    return max(0, mx - mn - 2 * k)

def ans_intervals(nums, k):
    L = max(a - k for a in nums)
    R = min(a + k for a in nums)
    return 0 if L <= R else L - R

def ans_min_k_for_zero(nums):
    return (max(nums) - min(nums) + 1) // 2

def ans_build_best_array(nums, k):
    mn, mx = min(nums), max(nums)
    if mx - mn <= 2 * k:
        return [mx - k] * len(nums)
    lo, hi = mn + k, mx - k
    return [min(max(a, lo), hi) for a in nums]

assert ans_smallest_range([3, 15, 8, 1], 4) == 6 and ans_build_best_array([3, 15, 8, 1], 4) == [5, 11, 8, 5]
assert list(range(5 - 2, 5 + 2)) == [3, 4, 5, 6] and max(0, 3 - 10) == 0 and min(max(12, 4), 9) == 9
assert ans_smallest_range([0, 10], 2) == 6 and ans_smallest_range([1, 3, 6], 3) == 0
assert ans_intervals([4, 9, 1, 7], 1) == 6
assert ans_min_k_for_zero([1, 3, 6]) == 3 and ans_min_k_for_zero([0, 10]) == 5 and ans_min_k_for_zero([7]) == 0
for nums in ([1, 3, 6], [0, 10], [2, 2, 9]):
    k0 = ans_min_k_for_zero(nums)
    assert Solution().smallestRangeI(nums, k0) == 0 and (k0 == 0 or Solution().smallestRangeI(nums, k0 - 1) > 0)
new = ans_build_best_array([4, 9, 1, 7], 1)
assert all(abs(x - a) <= 1 for x, a in zip(new, [4, 9, 1, 7])) and max(new) - min(new) == 6
print("Đáp án tham khảo PASS toàn bộ test! Mảng xây dựng:", new)

Đáp án tham khảo PASS toàn bộ test! Mảng xây dựng: [4, 8, 2, 7]


## 19. Template nhớ nhanh và checklist submit

```python
class Solution:
    def smallestRangeI(self, nums, k):
        # kéo min lên +k, kéo max xuống -k; không thể âm
        return max(0, max(nums) - min(nums) - 2 * k)
```

**Câu nhớ:** *"Chỉ hai đầu quan trọng: thu hẹp 2k, đừng xuống dưới 0."*

**Checklist:**
- [ ] Trừ **2k** (cả hai đầu đều được chỉnh), không phải `k`?
- [ ] Kẹp kết quả với `0`?
- [ ] Không nhầm với 910 (bắt buộc ±k)?
- [ ] Nếu tự tìm min/max: khởi tạo bằng `nums[0]`, không bằng 0?
- [ ] Test `k = 0`, mảng 1 phần tử, trường hợp `mx − mn = 2k` (vừa khít)?